In [2]:
import pandas as pd
import json

ledger = pd.read_csv("/content/ledger (1).csv")
gateway = pd.read_csv("/content/gateway (1).csv")

print("Ledger Data:")
print(ledger.head())

print("\nGateway Data:")
print(gateway.head())

Ledger Data:
  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       850.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R004       2026-03-02        M003      2100.0  success   
4           R005       2026-03-03        M004      7200.0  success   

  payment_method  
0            UPI  
1           Card  
2         Wallet  
3           Card  
4           Card  

Gateway Data:
  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       900.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R005       2026-03-03        M004      7200.0   failed   
4           R006       2026-03-03        M002       950.0  success   

  payment_method

In [3]:
print("Ledger Duplicates:", ledger.duplicated().sum())
print("Gateway Duplicates:", gateway.duplicated().sum())

print("\nLedger Null Values:")
print(ledger.isnull().sum())

print("\nGateway Null Values:")
print(gateway.isnull().sum())

Ledger Duplicates: 0
Gateway Duplicates: 0

Ledger Null Values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

Gateway Null Values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [4]:
print("Ledger Duplicates:", ledger.duplicated().sum())
print("Gateway Duplicates:", gateway.duplicated().sum())

print("\nLedger Null Values:")
print(ledger.isnull().sum())

print("\nGateway Null Values:")
print(gateway.isnull().sum())

Ledger Duplicates: 0
Gateway Duplicates: 0

Ledger Null Values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

Gateway Null Values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [6]:
missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway['transaction_id'])]
missing_in_gateway.to_csv("missing_in_gateway.csv", index=False)
print("missing_in_gateway.csv saved")

missing_in_gateway.csv saved


In [7]:
missing_in_ledger = gateway[
    ~gateway['transaction_id'].isin(ledger['transaction_id'])
]

print(missing_in_ledger.head())
print("Total missing in ledger:", len(missing_in_ledger))

  transaction_id transaction_date merchant_id  amount_usd   status  \
8           R011       2026-03-05        M003      1800.0  success   

  payment_method  
8           Card  
Total missing in ledger: 1


In [8]:
missing_in_ledger.to_csv("missing_in_ledger.csv", index=False)
print("missing_in_ledger.csv saved")

missing_in_ledger.csv saved


In [9]:
merged = pd.merge(
    ledger,
    gateway,
    on="transaction_id",
    suffixes=("_ledger", "_gateway")
)

print(merged.head())

  transaction_id transaction_date_ledger merchant_id_ledger  \
0           R001              2026-03-01               M001   
1           R002              2026-03-01               M002   
2           R003              2026-03-02               M001   
3           R005              2026-03-03               M004   
4           R006              2026-03-03               M002   

   amount_usd_ledger status_ledger payment_method_ledger  \
0             1200.0       success                   UPI   
1              850.0       success                  Card   
2              500.0       success                Wallet   
3             7200.0       success                  Card   
4              950.0       success                   UPI   

  transaction_date_gateway merchant_id_gateway  amount_usd_gateway  \
0               2026-03-01                M001              1200.0   
1               2026-03-01                M002               900.0   
2               2026-03-02                M001    

In [10]:
amount_mismatches = merged[
    merged["amount_usd_ledger"] != merged["amount_usd_gateway"]
]

print(amount_mismatches.head())
print("Total amount mismatches:", len(amount_mismatches))

  transaction_id transaction_date_ledger merchant_id_ledger  \
1           R002              2026-03-01               M002   
6           R008              2026-03-04               M001   

   amount_usd_ledger status_ledger payment_method_ledger  \
1              850.0       success                  Card   
6              640.0       success                  Card   

  transaction_date_gateway merchant_id_gateway  amount_usd_gateway  \
1               2026-03-01                M002               900.0   
6               2026-03-04                M001               600.0   

  status_gateway payment_method_gateway  
1        success                   Card  
6        success                   Card  
Total amount mismatches: 2


In [11]:
amount_mismatches.to_csv("amount_mismatches.csv", index=False)
print("amount_mismatches.csv saved")

amount_mismatches.csv saved


In [12]:
status_mismatches = merged[
    merged["status_ledger"] != merged["status_gateway"]
]

print(status_mismatches.head())
print("Total status mismatches:", len(status_mismatches))

  transaction_id transaction_date_ledger merchant_id_ledger  \
3           R005              2026-03-03               M004   

   amount_usd_ledger status_ledger payment_method_ledger  \
3             7200.0       success                  Card   

  transaction_date_gateway merchant_id_gateway  amount_usd_gateway  \
3               2026-03-03                M004              7200.0   

  status_gateway payment_method_gateway  
3         failed                   Card  
Total status mismatches: 1


In [13]:
status_mismatches.to_csv("status_mismatches.csv", index=False)
print("status_mismatches.csv saved")

status_mismatches.csv saved


In [14]:
reconciliation_report = {
    "missing_in_gateway": len(missing_in_gateway),
    "missing_in_ledger": len(missing_in_ledger),
    "amount_mismatches": len(amount_mismatches),
    "status_mismatches": len(status_mismatches)
}

report_df = pd.DataFrame([reconciliation_report])

print(report_df)

   missing_in_gateway  missing_in_ledger  amount_mismatches  status_mismatches
0                   2                  1                  2                  1


In [15]:
report_df.to_csv("reconciliation_report.csv", index=False)
print("reconciliation_report.csv saved")

reconciliation_report.csv saved


In [16]:
import json

with open("summary_metrics.json", "w") as f:
    json.dump(reconciliation_report, f, indent=4)

print("summary_metrics.json saved")

summary_metrics.json saved
